# Визуальные (мультимодальные / VLM) задачи

Здесь задача — это **изображение + вопрос**, а ответ остаётся текстовым, поэтому вся
проверка (`verify`) работает как обычно. Информация нужна для ответа считывается именно
с картинки (в тексте формулы/значений нет).

**Доступные типы (12, два бэкенда рендеринга — matplotlib и PIL):**

*Чтение графиков и диаграмм (matplotlib):*
- `function_plot_read` — прочитать график: число корней / f(0);
- `bar_chart_read` — столбчатая диаграмма: максимум/минимум/разница/сумма;
- `line_plot_read` — линейный график: значение в точке / x максимума / число возрастаний;
- `pie_chart_read` — круговая диаграмма: наибольший/наименьший сектор;
- `geometry_figure` — геометрия: площадь/периметр/неизвестный угол по подписям на рисунке.

*Счёт и распознавание (PIL):*
- `grid_color_count` — цветная сетка: посчитать клетки / самый частый цвет;
- `clock_read` — аналоговые часы: определить время (Ч:ММ);
- `dice_read` — игральные кости: сумма очков / сколько показывают значение;
- `chessboard_count` — фигуры на доске: всего / на тёмных / на светлых клетках.

*Картиночные варианты головоломок (логика наследуется от текстовых задач):*
- `sudoku_image` — судоку-картинка: какое число в выделенной клетке;
- `sliding_puzzle_image` — пятнашки-картинка: разрешимость / минимум ходов;
- `arc_grid_image` — ARC-индукция на цветных сетках «вход → выход».

*Графы, схемы и головоломки-2 (networkx / matplotlib / PIL):*
- `shortest_path_image` — взвешенный граф: вес кратчайшего пути;
- `mst_image` — взвешенный граф: вес минимального остовного дерева;
- `dfa_image` — диаграмма ДКА: принимает ли автомат строку;
- `boolean_circuit_image` — схема логических вентилей: значение выхода / число единичных наборов;
- `minesweeper_image` — поле «Сапёра»: мина или безопасно;
- `queens_check_image` — ферзи на доске: корректна ли расстановка;
- `mastermind_image` — доска «Мастермайнда»: восстановить код по цветным подсказкам;
- `pv_cycle_image` — P–V диаграмма: работа газа за цикл (площадь контура);
- `shoelace_image` — многоугольник по вершинам: его площадь.

Реестр визуальных задач — `ALL_VISUAL_TASK_GENERATORS` (отдельно от текстового
`ALL_TASK_GENERATORS`). Генерация датасета — `DatasetGenerator.generate_vlm_dataset()`.
Лёгкая аугментация — `augment_image(img, seed=...)`.

## 1. Показать по одной задаче каждого типа (с картинкой)

In [ ]:
from IPython.display import display
from re_rl.tasks.visual.generators import ALL_VISUAL_TASK_GENERATORS
from re_rl.tasks.registry import registry


def demo(task_type, sub=None, language="ru", difficulty=6):
    gen = ALL_VISUAL_TASK_GENERATORS[task_type]
    task = gen(task_type=sub, language=language, difficulty=difficulty, reasoning_mode=True)
    r = task.get_result()
    print(f"[{task_type}" + (f" / {sub}" if sub else "") + f"]  сложность={difficulty}")
    print("Вопрос:", r["problem"])
    print("Ответ:", r["final_answer"])
    print("verify(свой ответ):",
          task.verify(f"<answer>{r['final_answer']}</answer>"))
    display(task.render_image())
    print("=" * 80)


demo("function_plot_read", "count_roots")
demo("bar_chart_read", "max_category")
demo("line_plot_read", "max_x")
demo("pie_chart_read", "largest")
demo("geometry_figure", "right_triangle_area")
demo("geometry_figure", "missing_angle")
demo("grid_color_count", "count_color")
demo("clock_read")
demo("dice_read", "sum")
demo("chessboard_count", "on_dark")
demo("sudoku_image")
demo("sliding_puzzle_image", "min_moves")
demo("arc_grid_image")
demo("shortest_path_image")
demo("mst_image")
demo("dfa_image")
demo("boolean_circuit_image", "evaluate")
demo("minesweeper_image")
demo("queens_check_image")
demo("mastermind_image")
demo("pv_cycle_image")
demo("shoelace_image")

## 1b. Аугментация изображений

`augment_image(img, seed=...)` даёт лёгкий поворот на белом фоне — полезно для
устойчивости VLM. Ответ (`verify`) текстовый, поэтому корректность не меняется.

In [ ]:
from re_rl.tasks.visual import augment_image

t = ALL_VISUAL_TASK_GENERATORS["dice_read"](task_type="sum", language="ru", difficulty=6)
img = t.render_image()
print("Оригинал:")
display(img)
print("Аугментация (поворот):")
display(augment_image(img, seed=7))
print("Ответ прежний:", t.final_answer, "| verify:", t.verify(f"<answer>{t.final_answer}</answer>"))

## 2. Генерация мультимодального (VLM) датасета

PNG сохраняются в `output_dir/images/`, а в JSONL пишется относительный путь и токен
`<image>` в начале `input`. Сохранять — с `validate=False` (поле `image` не входит в
текстовую схему).

In [ ]:
from re_rl.dataset_generator import DatasetGenerator

gen = DatasetGenerator(output_dir="datasets_vlm")
vlm = gen.generate_vlm_dataset(
    task_types=None,          # все визуальные типы
    num_samples=30,
    language="ru",
    difficulties=[3, 5, 7],
    reasoning_mode=True,
    show_progress=True,
)

print(f"Сгенерировано: {len(vlm)} примеров")
print("=" * 70)
ex = vlm[0]
print("image :", ex["image"])
print("input :", ex["input"][:160])
print("output:", ex["output"][:300])

# Сохранение (validate=False из-за поля image)
gen.save_jsonl(vlm, "vlm_sample.jsonl", validate=False)